# Training Experiments
Compare training runs and hyperparameter sweeps. Load `results.csv` from each experiment and plot side-by-side.


In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import yaml

RUNS_DIR = Path('../runs')

all_csvs = sorted(RUNS_DIR.rglob('results.csv'), reverse=True)
print(f'Found {len(all_csvs)} training runs:')
for p in all_csvs:
    print(' -', p.parent.parent.name)

In [ ]:
# Load and compare all runs
dfs = {}
for csv_path in all_csvs:
    name = csv_path.parent.parent.name
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    dfs[name] = df

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, df in dfs.items():
    if 'metrics/mAP50(B)' in df.columns:
        axes[0].plot(df['metrics/mAP50(B)'], label=name)
    if 'train/box_loss' in df.columns:
        axes[1].plot(df['train/box_loss'], label=name)

axes[0].set_title('mAP@0.5 over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].legend(fontsize=8)

axes[1].set_title('Training Box Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Summary table: best mAP50 per run
summary_rows = []
for name, df in dfs.items():
    row = {'experiment': name}
    if 'metrics/mAP50(B)' in df.columns:
        row['best_mAP50'] = df['metrics/mAP50(B)'].max()
        row['best_epoch'] = df['metrics/mAP50(B)'].idxmax()
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values('best_mAP50', ascending=False)
print(summary_df.to_string(index=False))